# Python Tetris in Jupyter Notebook

`ipywidgets`로 만든 간단한 테트리스입니다.

## 실행 방법
1. 아래 설치 셀을 실행합니다.
2. 게임 코드 셀을 실행합니다.
3. `Start` 버튼을 누르면 게임이 시작됩니다.

## 조작
- `←` : 왼쪽 이동
- `→` : 오른쪽 이동
- `↓` : 한 칸 아래로 이동
- `⟳` : 회전
- `Drop` : 즉시 떨어뜨리기
- `Pause` : 일시정지
- `Restart` : 재시작

> 노트북 환경에서는 키보드 이벤트가 잘 안 잡히는 경우가 많아서 버튼 방식으로 만들었습니다.


In [ ]:
# 필요하면 먼저 실행하세요.
# 이미 ipywidgets가 설치되어 있으면 실행하지 않아도 됩니다.

%pip install ipywidgets


## 게임 실행

아래 셀을 실행하면 테트리스가 나타납니다.


In [ ]:

import random
import threading
import time
from IPython.display import display, HTML, clear_output
import ipywidgets as widgets

class JupyterTetris:
    WIDTH = 10
    HEIGHT = 20

    SHAPES = {
        "I": [[1, 1, 1, 1]],
        "O": [[1, 1],
              [1, 1]],
        "T": [[0, 1, 0],
              [1, 1, 1]],
        "S": [[0, 1, 1],
              [1, 1, 0]],
        "Z": [[1, 1, 0],
              [0, 1, 1]],
        "J": [[1, 0, 0],
              [1, 1, 1]],
        "L": [[0, 0, 1],
              [1, 1, 1]],
    }

    COLORS = {
        "I": "#7dd3fc",
        "O": "#fde047",
        "T": "#c084fc",
        "S": "#86efac",
        "Z": "#fca5a5",
        "J": "#93c5fd",
        "L": "#fdba74",
        "ghost": "#374151",
        "empty": "#111827",
        "grid": "#1f2937",
    }

    def __init__(self):
        self.output = widgets.Output()
        self.score_label = widgets.HTML()
        self.status_label = widgets.HTML()

        self.btn_start = widgets.Button(description="Start", button_style="success")
        self.btn_left = widgets.Button(description="←")
        self.btn_right = widgets.Button(description="→")
        self.btn_down = widgets.Button(description="↓")
        self.btn_rotate = widgets.Button(description="⟳")
        self.btn_drop = widgets.Button(description="Drop", button_style="warning")
        self.btn_pause = widgets.Button(description="Pause")
        self.btn_restart = widgets.Button(description="Restart", button_style="info")

        self.btn_start.on_click(lambda _: self.start())
        self.btn_left.on_click(lambda _: self.move(-1, 0))
        self.btn_right.on_click(lambda _: self.move(1, 0))
        self.btn_down.on_click(lambda _: self.soft_drop())
        self.btn_rotate.on_click(lambda _: self.rotate())
        self.btn_drop.on_click(lambda _: self.hard_drop())
        self.btn_pause.on_click(lambda _: self.toggle_pause())
        self.btn_restart.on_click(lambda _: self.restart())

        self.controls = widgets.VBox([
            widgets.HBox([self.btn_start, self.btn_pause, self.btn_restart]),
            widgets.HBox([self.btn_left, self.btn_right, self.btn_down, self.btn_rotate, self.btn_drop]),
            self.score_label,
            self.status_label,
        ])

        self.running = False
        self.paused = False
        self.lock = threading.Lock()
        self.thread = None
        self.reset_game()

    def reset_game(self):
        self.board = [[None for _ in range(self.WIDTH)] for _ in range(self.HEIGHT)]
        self.score = 0
        self.lines = 0
        self.level = 1
        self.game_over = False
        self.current = None
        self.next_piece = self.new_piece()
        self.spawn_piece()
        self.update_labels()
        self.draw()

    def new_piece(self):
        name = random.choice(list(self.SHAPES.keys()))
        shape = [row[:] for row in self.SHAPES[name]]
        return {
            "name": name,
            "shape": shape,
            "x": self.WIDTH // 2 - len(shape[0]) // 2,
            "y": 0,
        }

    def spawn_piece(self):
        self.current = self.next_piece
        self.current["x"] = self.WIDTH // 2 - len(self.current["shape"][0]) // 2
        self.current["y"] = 0
        self.next_piece = self.new_piece()
        if self.collides(self.current["shape"], self.current["x"], self.current["y"]):
            self.game_over = True
            self.running = False

    def collides(self, shape, x, y):
        for row_i, row in enumerate(shape):
            for col_i, cell in enumerate(row):
                if not cell:
                    continue
                bx = x + col_i
                by = y + row_i
                if bx < 0 or bx >= self.WIDTH or by >= self.HEIGHT:
                    return True
                if by >= 0 and self.board[by][bx] is not None:
                    return True
        return False

    def merge_piece(self):
        shape = self.current["shape"]
        name = self.current["name"]
        for row_i, row in enumerate(shape):
            for col_i, cell in enumerate(row):
                if cell:
                    bx = self.current["x"] + col_i
                    by = self.current["y"] + row_i
                    if 0 <= by < self.HEIGHT and 0 <= bx < self.WIDTH:
                        self.board[by][bx] = name

    def clear_lines(self):
        new_board = []
        cleared = 0

        for row in self.board:
            if all(cell is not None for cell in row):
                cleared += 1
            else:
                new_board.append(row)

        while len(new_board) < self.HEIGHT:
            new_board.insert(0, [None for _ in range(self.WIDTH)])

        self.board = new_board

        if cleared:
            self.lines += cleared
            self.score += [0, 100, 300, 500, 800][cleared] * self.level
            self.level = self.lines // 10 + 1

    def move(self, dx, dy):
        with self.lock:
            if self.game_over or self.paused:
                return
            new_x = self.current["x"] + dx
            new_y = self.current["y"] + dy
            if not self.collides(self.current["shape"], new_x, new_y):
                self.current["x"] = new_x
                self.current["y"] = new_y
            self.draw()

    def soft_drop(self):
        with self.lock:
            if self.game_over or self.paused:
                return
            if not self.collides(self.current["shape"], self.current["x"], self.current["y"] + 1):
                self.current["y"] += 1
                self.score += 1
            else:
                self.merge_piece()
                self.clear_lines()
                self.spawn_piece()
            self.update_labels()
            self.draw()

    def hard_drop(self):
        with self.lock:
            if self.game_over or self.paused:
                return
            distance = 0
            while not self.collides(self.current["shape"], self.current["x"], self.current["y"] + 1):
                self.current["y"] += 1
                distance += 1
            self.score += distance * 2
            self.merge_piece()
            self.clear_lines()
            self.spawn_piece()
            self.update_labels()
            self.draw()

    def rotate_shape(self, shape):
        return [list(row) for row in zip(*shape[::-1])]

    def rotate(self):
        with self.lock:
            if self.game_over or self.paused:
                return
            rotated = self.rotate_shape(self.current["shape"])

            # 벽 근처에서 회전할 때 약간 보정
            for offset in [0, -1, 1, -2, 2]:
                if not self.collides(rotated, self.current["x"] + offset, self.current["y"]):
                    self.current["shape"] = rotated
                    self.current["x"] += offset
                    break
            self.draw()

    def ghost_y(self):
        y = self.current["y"]
        while not self.collides(self.current["shape"], self.current["x"], y + 1):
            y += 1
        return y

    def board_with_piece(self):
        temp = [row[:] for row in self.board]

        # ghost piece
        gy = self.ghost_y()
        for row_i, row in enumerate(self.current["shape"]):
            for col_i, cell in enumerate(row):
                if cell:
                    bx = self.current["x"] + col_i
                    by = gy + row_i
                    if 0 <= by < self.HEIGHT and 0 <= bx < self.WIDTH and temp[by][bx] is None:
                        temp[by][bx] = "ghost"

        # current piece
        for row_i, row in enumerate(self.current["shape"]):
            for col_i, cell in enumerate(row):
                if cell:
                    bx = self.current["x"] + col_i
                    by = self.current["y"] + row_i
                    if 0 <= by < self.HEIGHT and 0 <= bx < self.WIDTH:
                        temp[by][bx] = self.current["name"]

        return temp

    def make_html(self):
        board = self.board_with_piece()
        html = """
        <style>
            .tetris-wrap {
                display: inline-block;
                padding: 12px;
                background: #020617;
                border-radius: 12px;
                border: 1px solid #334155;
                font-family: sans-serif;
            }
            .tetris-grid {
                display: grid;
                grid-template-columns: repeat(10, 24px);
                grid-template-rows: repeat(20, 24px);
                gap: 2px;
                background: #0f172a;
                padding: 6px;
                border-radius: 8px;
            }
            .tetris-cell {
                width: 24px;
                height: 24px;
                border-radius: 4px;
                box-sizing: border-box;
                border: 1px solid #1f2937;
            }
            .next-box {
                margin-top: 10px;
                color: #e5e7eb;
                font-size: 14px;
            }
        </style>
        <div class="tetris-wrap">
            <div class="tetris-grid">
        """

        for row in board:
            for cell in row:
                color = self.COLORS["empty"] if cell is None else self.COLORS[cell]
                opacity = "0.35" if cell == "ghost" else "1.0"
                html += f'<div class="tetris-cell" style="background:{color}; opacity:{opacity};"></div>'

        html += "</div>"
        html += f"<div class='next-box'>Next: <b>{self.next_piece['name']}</b></div>"
        html += "</div>"
        return html

    def draw(self):
        with self.output:
            clear_output(wait=True)
            display(HTML(self.make_html()))

    def update_labels(self):
        self.score_label.value = f"""
        <b>Score:</b> {self.score} &nbsp; 
        <b>Lines:</b> {self.lines} &nbsp; 
        <b>Level:</b> {self.level}
        """
        if self.game_over:
            self.status_label.value = "<b style='color:#ef4444;'>Game Over</b>"
        elif self.paused:
            self.status_label.value = "<b style='color:#f59e0b;'>Paused</b>"
        elif self.running:
            self.status_label.value = "<b style='color:#22c55e;'>Running</b>"
        else:
            self.status_label.value = "<b>Ready</b>"

    def speed(self):
        return max(0.08, 0.6 - (self.level - 1) * 0.04)

    def game_loop(self):
        while self.running and not self.game_over:
            time.sleep(self.speed())
            if self.paused:
                continue
            self.soft_drop()
        self.update_labels()
        self.draw()

    def start(self):
        if self.running:
            return
        if self.game_over:
            self.reset_game()
        self.running = True
        self.paused = False
        self.update_labels()
        self.thread = threading.Thread(target=self.game_loop, daemon=True)
        self.thread.start()

    def toggle_pause(self):
        if not self.running or self.game_over:
            return
        self.paused = not self.paused
        self.update_labels()
        self.draw()

    def restart(self):
        self.running = False
        time.sleep(0.05)
        self.paused = False
        self.reset_game()
        self.update_labels()

    def show(self):
        display(self.controls)
        display(self.output)
        self.draw()


game = JupyterTetris()
game.show()


## 추가 미션

코딩 연습용으로 다음 기능을 직접 추가해보면 좋습니다.

1. 키보드 조작 추가하기  
2. 다음 블록 미리보기 모양까지 표시하기  
3. 최고 점수 저장하기  
4. 난이도별 속도 조절하기  
5. 블록 색상 바꾸기  
